# 7_canli_tr — Canlı Çalıştırma, İzleme ve Tanı

Bu notebook, `tr_agent_zinciri.py` (zincir) ve `tr_veri_toplayici.py` (canlı veri) modüllerini kullanıyor. Zincirin mantığı notebook'ta değil, modüllerde. Bu notebook yalnızca çalıştırıyor, kaydediyor ve karşılaştırıyor.

| Bölüm | İş | Ne zaman |
|---|---|---|
| **A** | Kurulum ve ön kontroller | Her çalıştırmada |
| **B** | **Yarın için canlı karar**: EPİAŞ + Open-Meteo → zincir → Agent; sonuç kaydediliyor | Her gün **16:30'dan sonra** |
| **C** | Resmî planın yayın saatinin takibi | İsteğe bağlı |
| **D** | **Canlı tahminlerin gerçekleşenle karşılaştırılması** (daha önce kaydedilmiş günler) | Tahmin edilen günden sonraki gün, 02:00'den sonra |
| **E** | **28 Temmuz 2025 tanı deneyi**: fazla tahminin ne kadarı hava tahmininden geliyor? | Bir kez |

**Zamanlama:**
- **EAK:** Katılımcılar ertesi gün için bildirimlerini 14:00-16:00 arasında yapıyor.
- **Resmî plan:** G-1'de yayımlanıyor; saati belirtilmemiş.
- **Gerçekleşen tüketim:** 2 saat gecikmeyle yayımlanıyor.

B bölümü daha erken çalıştırılırsa zincir çökmüyor. Plan veya EAK eksikse yedek model devreye giriyor ve bu durum uyarı olarak raporlanıyor.

## A. Kurulum ve Ön Kontroller

In [1]:
import os
import sys
import json
import glob
import numpy as np
import pandas as pd
from dotenv import load_dotenv

KOK = r"C:\Deep_Learning_to_AI_Agents\Bitirme_Proje_Bonus"
load_dotenv(os.path.join(KOK, "notepads", ".env"), override=True)   # .env degistiyse (orn. yeni sifre) eski degerin uzerine yaz
os.environ["LANGSMITH_PROJECT"] = "tr-grid-agent"
sys.path.insert(0, os.path.join(KOK, "notepads"))
for m in ["tr_agent_zinciri", "tr_veri_toplayici"]:
    sys.modules.pop(m, None)                      # modul guncellendiyse yeniden yukle
from tr_agent_zinciri import Zincir, tekrar_oynatma_verisi, sonuc_json, n8n_yuku, kisisel_bilgi_temizle
import tr_veri_toplayici as tv

AGENT_CAGIR = True
simdi = pd.Timestamp.now(tz="Europe/Istanbul")
D = (simdi.normalize() + pd.Timedelta(days=1)).tz_localize(None)
z = Zincir(KOK, agent_cagir=AGENT_CAGIR)
# Golge calistirma: en son arsivlenen (onceki) model de her gun calistirilir; karar vermez, yalnizca karsilastirma icin
arsivler = sorted(glob.glob(os.path.join(KOK, "agent", "arsiv", "*")))
z_eski = Zincir(KOK, agent_cagir=False, model_klasoru=arsivler[-1]) if arsivler else None
print("Uretim modeli surumu:", z.surum, "| golge (onceki) model:", (z_eski.surum + " <- " + os.path.basename(arsivler[-1])) if z_eski else "yok")
print(f"Simdi: {simdi:%Y-%m-%d %H:%M} (Istanbul) | tahmin edilecek gun (D): {D.date()}")
print("Secilen model:", z.meta_secilen["model"], "| esikler:", {k: z.esik[k] for k in ["cok_nadir_alti", "nadir_alti"]})

tk = pd.read_csv(os.path.join(KOK, "data", "tr_tatil_takvimi.csv"), parse_dates=["yerel_gun"])
print("Tatil takvimi bitis:", tk["yerel_gun"].max().date(),
      "| kalan gun:", (tk["yerel_gun"].max() - D).days, "(bitmeden once 2_ozellik.ipynb'de yeni yil eklenmeli)")
if simdi.hour < 16 or (simdi.hour == 16 and simdi.minute < 30):
    print("UYARI: Saat 16:30'dan once. EAK bildirimi ve resmi plan henuz tamamlanmamis olabilir; "
          "yedek model devreye girebilir. Mumkunse 16:30'dan sonra calistirin.")

Uretim modeli surumu: 3d_uretim_20261002_1617 | golge (onceki) model: 3c (ilk secim) <- 20261002_1617
Simdi: 2026-10-06 02:34 (Istanbul) | tahmin edilecek gun (D): 2026-10-07
Secilen model: Hibrit (oran) + Open-Meteo | esikler: {'cok_nadir_alti': 0.207, 'nadir_alti': 0.246}
Tatil takvimi bitis: 2026-12-31 | kalan gun: 85 (bitmeden once 2_ozellik.ipynb'de yeni yil eklenmeli)
UYARI: Saat 16:30'dan once. EAK bildirimi ve resmi plan henuz tamamlanmamis olabilir; yedek model devreye girebilir. Mumkunse 16:30'dan sonra calistirin.


### A2. EPİAŞ oturum ve EAK erişim teşhisi

İlk canlı çalıştırmada yarının EAK'si alınamadı. Bu hücre EAK servisini **dün, bugün ve yarın** için birer kez, yeniden denemeden çağırıyor ve hata varsa türünü ve tam metnini yazdırıyor. Amaç, yarının EAK'sının neden alınamadığını kesinleştirmek:
- Hata 429 ise istek sınırı aşılmış.
- Hata veya boş yanıt yalnızca **yarın** için oluyor ve 16:00'dan sonra da sürüyorsa, EAK Şeffaflık Platformu'nda ertesi gün için ancak daha geç yayımlanıyor olabilir.
- Üç günde de hata varsa, erişim veya kimlik bilgisi sorunu var.

Sonuçlar `data/tr_eak_yayin_kaydi.csv`'de de birikiyor. B bölümü, yarının EAK'si alınamazsa artık durmuyor: D-1'in EAK profilini **yaklaşık** olarak kullanıyor ve kararı "geçici" uyarısıyla üretiyor.

**Önce oturum testi yapılıyor.** Giriş bilgileri reddedilirse (HTTP 401), hücre nedeni ve çözümü yazdırıp duruyor. Şifre değiştiyse `notepads\.env` içindeki `EPTR_PASSWORD`'u güncelleyin. Notebook `.env`'yi `override=True` ile okuduğu için kernel'i yeniden başlatmadan A hücresini tekrar çalıştırmak yeterli. Özel karakter içeren şifreyi tek tırnak içine alın: `EPTR_PASSWORD='...'`.

In [2]:
TESHIS = True
try:
    ist = {"eptr": tv.epias_istemci()}
    print("EPIAS oturumu: tamam")
except tv.EpiasHatasi as ex:
    print("EPIAS OTURUMU ACILAMADI:", ex)
    TESHIS = False
if TESHIS:
    for etiket, g in [("dun", D - pd.Timedelta(days=2)), ("bugun", D - pd.Timedelta(days=1)), ("yarin", D)]:
        try:
            d = ist["eptr"].call("eak", start_date=str(g.date()), end_date=str(g.date()), region="TR1")
            n = 0 if d is None else len(d)
            print(f"{etiket:6s} {g.date()}: {n} satir" + (f" | sutunlar: {list(d.columns)[:6]}" if n else " | BOS yanit"))
        except Exception as ex:
            print(f"{etiket:6s} {g.date()}: HATA {tv.epias_hata_ozeti(ex)}")
    print(f"Kontrol zamani: {pd.Timestamp.now(tz='Europe/Istanbul'):%H:%M} (Istanbul)")

EPIAS oturumu: tamam
dun    2026-10-03: 24 satir | sutunlar: ['date', 'time', 'toplam', 'dogalgaz', 'ruzgar', 'linyit']
bugun  2026-10-04: 24 satir | sutunlar: ['date', 'time', 'toplam', 'dogalgaz', 'ruzgar', 'linyit']
yarin  2026-10-05: HATA 400 BAD_REQUEST | (BUS)SEF1115: İstekeki başlangıç tarihi(2026-10-05T00:00:00.000+0300) bitiş tarihinden(2026-10-04T00:00:00.000+0300) sonra olamaz!
Kontrol zamani: 02:03 (Istanbul)


### A3. Önceki çalıştırmalardan kalan kişisel bilgilerin temizlenmesi (bir kez)

İlk canlı çalıştırmada EPİAŞ'ın hata yanıtı, olduğu gibi veri uyarılarına ve kayıtlara taşınmıştı. Bu yanıt kullanıcı adını (e-posta) ve istemci IP adresini de içeriyordu. Bu bilgiler `data/canli/*/sonuc.json` ve `data/tr_eak_yayin_kaydi.csv` dosyalarına yazıldı, ayrıca Gemini'ye giden prompt'a ve LangSmith izine girdi.

Toplayıcı artık yalnızca hata kodunu ve açıklamasını taşıyor (`epias_hata_ozeti`). Bu hücre, **diskteki** eski kayıtlarda kalan e-posta ve IP adreslerini `[gizlendi]` ile değiştiriyor.

LangSmith'teki eski izin (`tr-grid-agent` projesi, 30 Eylül 2026 sabahki çalıştırma) web arayüzünden elle silinmeli. Gemini'ye gönderilmiş veri için yapılabilecek bir şey yok.

In [3]:
import re
KISISEL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+|\b\d{1,3}(?:\.\d{1,3}){3}\b")
say = {"n": 0}
def temizle(o):
    if isinstance(o, str):
        yeni, n = KISISEL.subn("[gizlendi]", o); say["n"] += n; return yeni
    if isinstance(o, list):
        return [temizle(x) for x in o]
    if isinstance(o, dict):
        return {k: temizle(v) for k, v in o.items()}
    return o
for dosya in glob.glob(os.path.join(KOK, "data", "canli", "*", "sonuc.json")):
    with open(dosya, encoding="utf-8") as f:
        s = json.load(f)
    once = say["n"]; s = temizle(s)
    if say["n"] > once:
        with open(dosya, "w", encoding="utf-8") as f:
            json.dump(s, f, ensure_ascii=False, indent=2)
        print("Temizlendi:", dosya)
for ad in ["tr_eak_yayin_kaydi.csv", "tr_plan_yayin_kaydi.csv"]:
    yol = os.path.join(KOK, "data", ad)
    if os.path.exists(yol):
        d = pd.read_csv(yol, dtype=str)
        once = say["n"]; d = d.apply(lambda s: s.map(lambda v: temizle(v) if isinstance(v, str) else v))
        if say["n"] > once:
            d.to_csv(yol, index=False); print("Temizlendi:", yol)
print("Gizlenen e-posta/IP sayisi:", say["n"])

Gizlenen e-posta/IP sayisi: 0


## B. Yarın İçin Canlı Karar

1. `canli_veri`: EPİAŞ'tan tüketim, plan ve EAK; Open-Meteo'dan 6 ilin tahmini ve sapma penceresi. Sonuçlar `data/canli/<D>/` klasörüne önbelleğe alınıyor.
2. `z.calistir(D, kaynak_bilgisi=bilgi)`: Zincir. Toplayıcının veri uyarıları Agent'a da gidiyor.
3. Sonuç `data/canli/<D>/sonuc.json` olarak kaydediliyor; D bölümü bu dosyayı ertesi gün gerçekleşenle karşılaştıracak.
4. **Gizlilik denetimi:** Kaydedilen sonuçta veya n8n yükünde e-posta ya da IP adresi bulunursa çalışma durur ve hiçbir şey gönderilmez. Bu, toplayıcıdaki ve zincirdeki temizlik katmanlarının arkasındaki son kontrol.

`YENILE = True`, önbelleği yok sayıp bütün verileri yeniden indirir. Plan veya EAK sonradan tamamlandıysa zaten yeniden isteniyor.
5. **Gölge çalıştırma:** `agent/arsiv/` altındaki en son arşivlenmiş (önceki) model de aynı veriyle çalıştırılıyor ve `sonuc_eski_model.json` olarak kaydediliyor. Bu model karar vermiyor ve Agent'ı çağırmıyor; yalnızca D bölümünde, üretim modeliyle gerçekleşen tüketim üzerinden karşılaştırılıyor. Yeni bir modeli, eskisi yanında karar vermeden çalıştırarak sınamak, model değişikliklerinde kullanılan standart yöntem.

In [4]:
YENILE = False
ham, bilgi = tv.canli_veri(KOK, D, z.meta_secilen, yenile=YENILE)
print("Veri kaynaklari:")
for k, v in bilgi["kaynak"].items():
    print(f"  {k:8s}: {v}")
z.veri(ham)
r = z.calistir(D, kaynak_bilgisi=bilgi)
o, c = r["ozet"], r["cikti"]
print(f"\nKARAR: {c['karar']} | durum {c['durum']} | oncelik {c['oncelik']} | model: {o['model']}")
print(f"En dar saat {o['en_dar_saat']:02d}:00 | marj %{o['en_dusuk_marj'] * 100:.1f} "
      f"(tahmin {o['tahmin_en_dar_saatte_mwh']} MWh, EAK {o['eak_en_dar_saatte_mwh']} MWh) | "
      f"temkinli marj %{o['en_dusuk_marj_temkinli'] * 100:.1f}")
print(f"Zirve tahmini {o['tahmin_zirve_mwh']} MWh, saat {o['tahmin_zirve_saati']:02d}:00 | sapma kaynagi: {o['sapma_kaynagi']}")
print("Veri uyarilari:", o["uyarilar"] or "yok")
print("Dogrulama notlari:", r["notlar"] or "yok")

[02:03:25] EPIAS: tuketim, plan ve EAK aliniyor...


  eak 2026-10-05-2026-10-05 deneme 1: 400 BAD_REQUEST | (BUS)SEF1115: İstekeki başlangıç tarihi(2026-10-05T00:00:00.000+0300) bitiş tarihinden(2026-10-04T00:00:00.000+0300) sonra olamaz!
  eak 2026-10-05-2026-10-05 deneme 2: 400 BAD_REQUEST | (BUS)SEF1115: İstekeki başlangıç tarihi(2026-10-05T00:00:00.000+0300) bitiş tarihinden(2026-10-04T00:00:00.000+0300) sonra olamaz!
[02:03:42] Open-Meteo: hava tahmini ve sapma penceresi aliniyor...
[02:03:56] Veri toplama tamam.
Veri kaynaklari:
  tuketim : rt-cons, son saat 2026-10-04 00:00:00+03:00
  plan    : load-plan, D icin 0/24 saat
  eak     : eak, D icin 0/24 saat -> YAKLASIK: D-1 profili kullanildi
  hava    : GFS D: 24/24 saat | sapma penceresi 2026-08-28 - 2026-09-26: tahmin 720, ERA5 720 saat

KARAR: normal_operation | durum normal | oncelik yok | model: Referans (3_model)
En dar saat 19:00 | marj %28.7 (tahmin 43281 MWh, EAK 60738 MWh) | temkinli marj %25.8
Zirve tahmini 43281 MWh, saat 19:00 | sapma kaynagi: None
Veri uyarilari: ['R

### B2. Saatlik profil, Agent açıklaması ve kayıt

In [5]:
print(r["saatlik"].to_string(index=False))
print("\nAgent ciktisi:")
print(json.dumps(c, ensure_ascii=False, indent=2))

kayit = sonuc_json(r)
kayit["model_surumu"] = z.surum
kayit["olusturma"] = simdi.isoformat()
kayit["toplayici_bilgisi"] = kisisel_bilgi_temizle(bilgi)
yol = os.path.join(KOK, "data", "canli", str(D.date()), "sonuc.json")
with open(yol, "w", encoding="utf-8") as f:
    json.dump(kayit, f, ensure_ascii=False, indent=2, default=str)
print("\nKaydedildi:", yol)

# Golge model: ayni ham veriyle (ayni plan, EAK, hava) onceki model; Agent cagrilmaz
if z_eski is not None:
    z_eski.veri(ham)
    r_e = z_eski.calistir(D, kaynak_bilgisi=bilgi, agent=False)
    kayit_e = sonuc_json(r_e); kayit_e["model_surumu"] = z_eski.surum; kayit_e["olusturma"] = simdi.isoformat()
    yol_e = os.path.join(KOK, "data", "canli", str(D.date()), "sonuc_eski_model.json")
    with open(yol_e, "w", encoding="utf-8") as f:
        json.dump(kayit_e, f, ensure_ascii=False, indent=2, default=str)
    oe = r_e["ozet"]
    fark = (r["saatlik"]["tahmin_mwh"].values - r_e["saatlik"]["tahmin_mwh"].values) / r_e["saatlik"]["tahmin_mwh"].values * 100
    print(f"\nGOLGE KARSILASTIRMA ({z.surum} vs {z_eski.surum}):")
    print(f"  Uretim: karar {c['karar']}, en dar saat {o['en_dar_saat']:02d}:00, marj %{o['en_dusuk_marj'] * 100:.1f}, zirve {o['tahmin_zirve_mwh']} MWh")
    print(f"  Onceki: karar {r_e['cikti']['karar']}, en dar saat {oe['en_dar_saat']:02d}:00, marj %{oe['en_dusuk_marj'] * 100:.1f}, zirve {oe['tahmin_zirve_mwh']} MWh")
    print(f"  Saatlik fark (uretim - onceki): ortalama %{fark.mean():+.2f}, en buyuk %{fark[abs(fark).argmax()]:+.2f} (saat {int(abs(fark).argmax()):02d}:00)")
    print("  Kaydedildi:", yol_e)

# Gizlilik denetimi: kaydedilen sonuc ve n8n yuku e-posta veya IP adresi icermemeli (icerirse calisma durur)
import re
from tr_agent_zinciri import EPOSTA_DESENI, IPV4_DESENI
with open(yol, encoding="utf-8") as f:
    denetlenen = f.read() + json.dumps(n8n_yuku(r), ensure_ascii=False)
if z_eski is not None:
    with open(yol_e, encoding="utf-8") as f:
        denetlenen += f.read()
bulunan = re.findall(EPOSTA_DESENI, denetlenen) + re.findall(IPV4_DESENI, denetlenen)
assert not bulunan, f"GIZLILIK DENETIMI BASARISIZ: {len(bulunan)} kisisel bilgi bulundu; gonderim durduruldu"
print("Gizlilik denetimi: kaydedilen sonuc ve n8n yuku kisisel bilgi icermiyor.")

GONDER = False
url = os.getenv("N8N_TR_WEBHOOK_URL")
if GONDER and url:
    import requests
    print("n8n yaniti:", requests.post(url, json=n8n_yuku(r), timeout=30).status_code)
else:
    print("n8n'e gonderilmedi (GONDER = False veya N8N_TR_WEBHOOK_URL tanimli degil).")

 saat  tahmin_mwh  temkinli_mwh  eak_mwh   marj  marj_temkinli  resmi_plan_mwh
    0     31863.0       33129.0  61863.0 0.4849         0.4645             NaN
    1     30471.0       31633.0  61564.0 0.5050         0.4862             NaN
    2     29220.0       30319.0  61178.0 0.5224         0.5044             NaN
    3     28785.0       29849.0  60871.0 0.5271         0.5096             NaN
    4     28500.0       29554.0  60901.0 0.5320         0.5147             NaN
    5     29051.0       30134.0  61093.0 0.5245         0.5067             NaN
    6     30306.0       31536.0  61395.0 0.5064         0.4863             NaN
    7     33522.0       34963.0  63582.0 0.4728         0.4501             NaN
    8     40203.0       41970.0  67238.0 0.4021         0.3758             NaN
    9     42130.0       43920.0  70751.0 0.4045         0.3792             NaN
   10     42473.0       44488.0  73803.0 0.4245         0.3972             NaN
   11     42802.0       44843.0  75313.0 0.4317     

## C. Resmî Planın ve EAK'nın Yayın Saatinin Takibi

Toplayıcı her çalıştırmada D günü **EAK'sında** kaç saat olduğunu ve alınamadıysa hata mesajını da `data/tr_eak_yayin_kaydi.csv`'ye yazıyor. Aşağıdaki ikinci tablo bunu gösteriyor.

Toplayıcı her çalıştırmada D günü planında kaç saat olduğunu `data/tr_plan_yayin_kaydi.csv`'ye yazıyor. Bu tablo, planın gün içinde **ilk kez tam (24/24)** göründüğü kontrol zamanlarını gösteriyor.

`BEKLE = True` yapılırsa, plan tamamlanana kadar 30 dakikada bir, en fazla 6 kez (3 saat) yalnızca plan kontrol ediliyor. Her kontrol tek bir EPİAŞ isteği. Varsayılan `False`; böylece "Run All" beklemeden biter.

In [6]:
BEKLE = False
if BEKLE:
    import time
    istemci = {"eptr": tv.epias_istemci()}
    for deneme in range(6):
        p = tv._epias_df(istemci, "load-plan", D, D, ["lep"])
        n = int((p["date"].dt.tz_localize(None).dt.normalize() == D).sum()) if len(p) else 0
        tv.plan_yayin_kaydi(KOK, D, n)
        print(f"{pd.Timestamp.now(tz='Europe/Istanbul'):%H:%M} | D planinda {n}/24 saat")
        if n == 24:
            break
        time.sleep(30 * 60)

yk = os.path.join(KOK, "data", "tr_plan_yayin_kaydi.csv")
if os.path.exists(yk):
    k = pd.read_csv(yk, parse_dates=["kontrol_zamani"])
    k["kontrol_saati"] = k["kontrol_zamani"].dt.strftime("%H:%M")
    ozet_k = k.groupby("gun").apply(lambda g: pd.Series({
        "kontrol_sayisi": len(g),
        "ilk_tam_gorulen_kontrol": g.loc[g["plandaki_saat"] == 24, "kontrol_saati"].min(),
        "son_eksik_gorulen_kontrol": g.loc[g["plandaki_saat"] < 24, "kontrol_saati"].max()}), include_groups=False)
    print(ozet_k.fillna("-").to_string())
    print("\nYorum: Yayin saati, 'son eksik' ile 'ilk tam' kontrolleri arasindadir; ayni gun icin iki deger varsa aralik daralir.")

ye = os.path.join(KOK, "data", "tr_eak_yayin_kaydi.csv")
if os.path.exists(ye):
    e = pd.read_csv(ye, parse_dates=["kontrol_zamani"]).fillna({"hata": ""})
    e["kontrol_saati"] = e["kontrol_zamani"].dt.strftime("%H:%M")
    print("\nEAK yayin kaydi (D gunu icin kontroller):")
    print(e[["gun", "kontrol_saati", "eak_saat", "hata"]].assign(hata=e["hata"].str[-90:]).tail(15).to_string(index=False))

            kontrol_sayisi ilk_tam_gorulen_kontrol son_eksik_gorulen_kontrol
gun                                                                         
2026-10-01               3                       -                     10:15
2026-10-03               5                   16:05                         -
2026-10-04               1                   19:38                         -
2026-10-05               1                       -                     02:03

Yorum: Yayin saati, 'son eksik' ile 'ilk tam' kontrolleri arasindadir; ayni gun icin iki deger varsa aralik daralir.

EAK yayin kaydi (D gunu icin kontroller):
       gun kontrol_saati  eak_saat                                                                                       hata
2026-10-01         09:57         0 izlendi]","userName":"[gizlendi]","successMessage":null,"errors":[{"errorCode":"(BUS)SEF11
2026-10-01         10:15         0 2026-10-01T00:00:00.000+0300) bitiş tarihinden(2026-09-30T00:00:00.000+0300) sonra olamaz!

## D. Canlı Tahminlerin Gerçekleşenle Karşılaştırılması

`data/canli/*/sonuc.json` dosyalarındaki geçmiş günler için gerçekleşen tüketim EPİAŞ'tan alınıyor. Karşılaştırma yalnızca gerçekleşen tüketimi 24 saat tamamlanmış günler için yapılıyor. Her gün için:
- Modelin ve resmî planın günlük hatası (MAPE).
- En dar saat ve marj: tahminle ve gerçekleşenle, aynı EAK üzerinden.
- **Karar tutarlılığı:** Aynı kural, gerçekleşen tüketime uygulansaydı hangi karar çıkardı? Canlı karar bununla aynı mı?

Sonuçlar `metrics/tr_canli_izleme.csv`'ye yazılıyor. Günler biriktikçe, canlı performansın test yılı sonuçlarıyla (seçilen model %2,29, resmî plan %2,60) karşılaştırılması mümkün olacak.

**Gerçek plan ve gerçek EAK da alınıyor.** Kayıtta resmî plan eksik olabilir (plansız yedek modelin kullanıldığı günler), EAK da D-1 yaklaşımı olabilir (sabah yapılan çalıştırmalar). Bu yüzden karşılaştırma için o günün **gerçek** planı ve **gerçek** EAK'sı EPİAŞ'tan yeniden alınıyor:
- Planın hatası her gün ölçülüyor.
- Gerçekleşen marj gerçek EAK ile hesaplanıyor.
- `eak_hatasi_%`, kullanılan EAK'nın gerçek EAK'dan sapmasını gösteriyor. D-1 yaklaşımının kullanıldığı günlerde bu, yaklaşımın isabetinin ölçüsü.

In [2]:
def kural(marj_min):
    if marj_min <= 0:
        return "activate_reserve"
    if marj_min < z.esik["nadir_alti"]:
        return "demand_response_alert"
    return "normal_operation"

bugun = simdi.normalize().tz_localize(None)
satirlar, istemci = [], None
for dosya in sorted(glob.glob(os.path.join(KOK, "data", "canli", "*", "sonuc.json"))):
    with open(dosya, encoding="utf-8") as f:
        s = json.load(f)
    g = pd.Timestamp(s["ozet"]["gun"])
    if g >= bugun:
        continue                                   # henuz yasanmamis veya bugun
    istemci = istemci or {"eptr": tv.epias_istemci()}

    def gun_serisi(anahtar, sutun, **ek):
        try:
            d = tv._epias_df(istemci, anahtar, g, g, [sutun], **ek)
        except tv.EpiasHatasi as ex:
            print(f"  {g.date()} {anahtar}: {ex}"); return None
        d = d[d["date"].dt.tz_localize(None).dt.normalize() == g].sort_values("date")
        return d[sutun].values.astype(float) if len(d) == 24 else None

    y = gun_serisi("rt-cons", "consumption")
    if y is None:
        print(f"{g.date()}: gerceklesen tuketim henuz tam degil"); continue
    p_ger = gun_serisi("load-plan", "lep")                 # gercek resmi plan (kayitta eksik olabilir)
    e_ger = gun_serisi("eak", "toplam", region="TR1")       # gercek EAK (kayitta D-1 yaklasimi olabilir)
    sa = pd.DataFrame(s["saatlik"])
    yol_e = os.path.join(os.path.dirname(dosya), "sonuc_eski_model.json")
    eski_mape = None
    if os.path.exists(yol_e):
        with open(yol_e, encoding="utf-8") as f:
            t_e = pd.DataFrame(json.load(f)["saatlik"])["tahmin_mwh"].values
        eski_mape = round(float(np.mean(np.abs(t_e - y) / y) * 100), 2)
    t, e_kul, t_tem = sa["tahmin_mwh"].values, sa["eak_mwh"].values, sa["temkinli_mwh"].values
    p = p_ger if p_ger is not None else sa["resmi_plan_mwh"].values.astype(float)
    eak_kaynak = ((s.get("toplayici_bilgisi") or {}).get("kaynak") or {}).get("eak", "")
    e_ref = e_ger if e_ger is not None else e_kul
    marj_g = (e_ref - y) / e_ref
    satirlar.append({
        "gun": str(g.date()), "surum": s.get("model_surumu", "3c (ilk secim)"), "model": s["ozet"]["model"], "eak_kaynagi": "D-1 yaklasimi" if "YAKLASIK" in eak_kaynak else "gercek",
        "canli_karar": s["cikti"]["karar"], "gerceklesenle_karar": kural(marj_g.min()),
        "model_mape_%": round(float(np.mean(np.abs(t - y) / y) * 100), 2), "onceki_model_mape_%": eski_mape,
        "plan_mape_%": round(float(np.mean(np.abs(p - y) / y) * 100), 2) if np.isfinite(p).all() else None,
        "eak_hatasi_%": round(float(np.mean(np.abs(e_kul - e_ger) / e_ger) * 100), 2) if e_ger is not None else None,
        "tahmin_en_dar_saat": s["ozet"]["en_dar_saat"], "gercek_en_dar_saat": int(np.argmin(marj_g)),
        "tahmin_marj_%": round(s["ozet"]["en_dusuk_marj"] * 100, 1), "gercek_marj_%": round(float(marj_g.min()) * 100, 1),
        "temkinli_kapsama_%": round(float(np.mean(y <= t_tem)) * 100, 1)})
if satirlar:
    izl = pd.DataFrame(satirlar)
    izl["karar_tutarli"] = izl["canli_karar"] == izl["gerceklesenle_karar"]
    print(izl.to_string(index=False))
    ort = lambda c: izl[c].dropna().mean()
    print(f"\n{len(izl)} gun | model MAPE ort. %{ort('model_mape_%'):.2f} | plan MAPE ort. %{ort('plan_mape_%'):.2f} | "
          f"karar tutarliligi %{izl['karar_tutarli'].mean() * 100:.0f} | temkinli bant kapsamasi ort. %{ort('temkinli_kapsama_%'):.1f}")
    ab = izl.dropna(subset=["onceki_model_mape_%"])
    if len(ab):
        iyi = int((ab["model_mape_%"] < ab["onceki_model_mape_%"]).sum())
        print(f"GOLGE KARSILASTIRMA: {len(ab)} gun | uretim modeli MAPE ort. %{ab['model_mape_%'].mean():.2f} | "
              f"onceki model %{ab['onceki_model_mape_%'].mean():.2f} | uretim modeli {iyi}/{len(ab)} gunde daha iyi")
    vek = izl[izl["eak_kaynagi"] == "D-1 yaklasimi"]["eak_hatasi_%"].dropna()
    if len(vek):
        print(f"D-1 EAK yaklasiminin gercek EAK'ya gore ortalama hatasi: %{vek.mean():.2f} ({len(vek)} gun)")
    izl.to_csv(os.path.join(KOK, "metrics", "tr_canli_izleme.csv"), index=False)
else:
    print("Karsilastirilacak tamamlanmis gun yok. B bolumunu bir gun calistirin; ertesi gun 02:00'den sonra bu bolum o gunu karsilastirir.")

       gun                   surum                      model   eak_kaynagi      canli_karar gerceklesenle_karar  model_mape_%  onceki_model_mape_%  plan_mape_%  eak_hatasi_%  tahmin_en_dar_saat  gercek_en_dar_saat  tahmin_marj_%  gercek_marj_%  temkinli_kapsama_%  karar_tutarli
2026-10-01          3c (ilk secim)         Referans (3_model) D-1 yaklasimi normal_operation    normal_operation          1.42                  NaN         2.22          1.30                  19                  19           31.7           31.5                95.8           True
2026-10-03 3d_uretim_20261002_1617 Hibrit (oran) + Open-Meteo        gercek normal_operation    normal_operation          1.49                 2.99         1.39          0.68                  19                  19           37.1           34.9               100.0           True
2026-10-04 3d_uretim_20261002_1617 Hibrit (oran) + Open-Meteo        gercek normal_operation    normal_operation          2.16                 2.58         3.73

## E. Tanı Deneyi: 28 Temmuz 2025'teki Fazla Tahmin

`6_agent_tr`'de bu gün için model %4,15, resmî plan %1,55 hata yapmıştı. Model tüketimi öğleden sonra +%5-8 fazla tahmin etmişti. İki olası etken vardı:
1. GFS tahmininin o gün ~1°C fazla sıcak olması (30,80°C'ye karşı 29,67°C).
2. Modelin, doğru olan planı yanlış yönde "düzeltmesi".

**Deney:** Aynı gün üç şekilde çalıştırılıyor:
- (a) Normal: GFS tahmini, sapma düzeltmeli.
- (b) Kusursuz hava: GFS tahmini yerine gerçekleşen sıcaklık (ERA5); sapma düzeltmesi kapalı.
- (c) Resmî plan.

(a) ile (b) arasındaki fark **hava tahmininin** katkısını, (b)'de kalan fazla tahmin ise **modelin kendi düzeltmesinin** katkısını gösteriyor.

**E2 (genelleme):** Tek bir günden sonuç çıkarmamak için aynı deney, test yılının (Eki 2025 – Eyl 2026) **en sıcak 10 günü** için tekrarlanıyor. Günler, ERA5 ağırlıklı günlük ortalama sıcaklığa göre seçiliyor.

Bu bir tanı deneyi; modeli, seçimi veya kararları değiştirmiyor. Agent çağrılmıyor. 28 Temmuz 2025, seçilen modelin erken durdurma penceresinde (2025-04 → 2025-09); test yılındaki günler ise tam örneklem dışı.

In [8]:
zt = Zincir(KOK, agent_cagir=False)
zt.veri(tekrar_oynatma_verisi(KOK, zt.meta_secilen))
asil_sapma = zt.sapma_saatlik

def deney(gun):
    Dg = pd.Timestamp(gun)
    a = zt.gun_oncesi_tahmin(Dg)
    m = zt.df["gun_t"] == Dg
    yedek = zt.df.loc[m, "om_fc_ham"].copy()
    zt.df.loc[m, "om_fc_ham"] = zt.df.loc[m, "om_era5"].values          # (b) kusursuz hava
    zt.sapma_saatlik = lambda _g: (np.zeros(24), "kapali (ERA5)")
    try:
        b = zt.gun_oncesi_tahmin(Dg)
    finally:
        zt.df.loc[m, "om_fc_ham"] = yedek.values
        zt.sapma_saatlik = asil_sapma
    g = a["g"]
    y = g["consumption"].values
    ort = lambda v: float(np.mean((v - y) / y) * 100)
    mape = lambda v: float(np.mean(np.abs(v - y) / y) * 100)
    return {"gun": str(Dg.date()), "model_a": a["model_anahtar"],
            "gfs_ham_C": round(float(g["om_fc_ham"].mean()), 2), "gfs_duzeltilmis_C": round(float(a["g"]["om_t"].mean()), 2) if "om_t" in a["g"] else None,
            "era5_C": round(float(g["om_era5"].mean()), 2),
            "a_mape_%": round(mape(a["tahmin"]), 2), "b_mape_%": round(mape(b["tahmin"]), 2), "plan_mape_%": round(mape(g["lep_ham"].values), 2),
            "a_sapma_%": round(ort(a["tahmin"]), 2), "b_sapma_%": round(ort(b["tahmin"]), 2), "plan_sapma_%": round(ort(g["lep_ham"].values), 2)}, a, b

s28, a28, b28 = deney("2025-07-28")
print("28 Temmuz 2025:", json.dumps(s28, ensure_ascii=False))
y28 = a28["g"]["consumption"].values
tablo28 = pd.DataFrame({"saat": a28["g"]["saat"].values, "gercek": np.round(y28), "plan_%": np.round((a28["g"]["lep_ham"].values - y28) / y28 * 100, 2),
                        "a_GFS_%": np.round((a28["tahmin"] - y28) / y28 * 100, 2), "b_ERA5_%": np.round((b28["tahmin"] - y28) / y28 * 100, 2)})
print(tablo28.iloc[::2].to_string(index=False))
hava_katkisi = s28["a_sapma_%"] - s28["b_sapma_%"]
print(f"\nGunluk ortalama sapma: (a) %{s28['a_sapma_%']:+.2f} | (b) %{s28['b_sapma_%']:+.2f} | plan %{s28['plan_sapma_%']:+.2f}")
print(f"Hava tahmininin katkisi (a - b): {hava_katkisi:+.2f} puan | kusursuz havada bile kalan (b): {s28['b_sapma_%']:+.2f} puan")

28 Temmuz 2025: {"gun": "2025-07-28", "model_a": "secilen", "gfs_ham_C": 30.8, "gfs_duzeltilmis_C": 30.71, "era5_C": 29.67, "a_mape_%": 0.85, "b_mape_%": 0.35, "plan_mape_%": 1.55, "a_sapma_%": 0.76, "b_sapma_%": 0.02, "plan_sapma_%": 1.43}
 saat  gercek  plan_%  a_GFS_%  b_ERA5_%
    0 45663.0    3.95     1.07      0.29
    2 41972.0    3.41     0.86     -0.06
    4 39850.0    3.05     1.50      0.02
    6 38184.0    1.47     1.32     -0.03
    8 48165.0    1.12     1.20     -0.03
   10 55353.0   -0.95    -0.75     -1.33
   12 56630.0    1.36     0.56      0.03
   14 59504.0    0.88     1.35      0.87
   16 59131.0   -0.04     0.34     -0.01
   18 55729.0   -0.05    -0.05     -0.51
   20 55039.0    2.27     1.40      1.16
   22 53571.0    1.46     0.48     -0.10

Gunluk ortalama sapma: (a) %+0.76 | (b) %+0.02 | plan %+1.43
Hava tahmininin katkisi (a - b): +0.74 puan | kusursuz havada bile kalan (b): +0.02 puan


### E2. Genelleme: test yılının en sıcak 10 günü

In [9]:
test = zt.df[(zt.df["date"] >= pd.Timestamp("2025-10-01T00:00:00+03:00")) & (zt.df["date"] < pd.Timestamp("2026-09-23T00:00:00+03:00"))]
gunluk = test.groupby("gun_t").agg(era5=("om_era5", "mean"), n=("om_era5", "count"), fc=("om_fc_ham", "count"))
sicak = gunluk[(gunluk["n"] == 24) & (gunluk["fc"] == 24)].nlargest(10, "era5").index
satir = []
for g in sicak:
    try:
        satir.append(deney(g)[0])
    except Exception as ex:
        print(f"{g.date()}: atlandi ({str(ex)[:80]})")
e2 = pd.DataFrame(satir)
print(e2.to_string(index=False))
if len(e2):
    print(f"\n{len(e2)} sicak gun ortalamasi | MAPE: (a) %{e2['a_mape_%'].mean():.2f}, (b) %{e2['b_mape_%'].mean():.2f}, plan %{e2['plan_mape_%'].mean():.2f}")
    print(f"Ortalama sapma: (a) %{e2['a_sapma_%'].mean():+.2f}, (b) %{e2['b_sapma_%'].mean():+.2f}, plan %{e2['plan_sapma_%'].mean():+.2f}")
    print(f"GFS'in bu gunlerdeki ortalama sicaklik farki (duzeltilmis - ERA5): {(e2['gfs_duzeltilmis_C'] - e2['era5_C']).mean():+.2f} C")
    e2.to_csv(os.path.join(KOK, "metrics", "tr_sicak_gun_tanisi.csv"), index=False)
    pd.DataFrame([s28]).to_csv(os.path.join(KOK, "metrics", "tr_28temmuz_tanisi.csv"), index=False)
print("""
Yorum rehberi:
- (a) ile (b) arasindaki fark buyukse sorun agirlikla hava tahminindedir; sapma duzeltmesi iyilestirilmelidir.
- (b) de plandan belirgin kotuyse, model kusursuz havada bile plani yanlis yonde duzeltiyordur; duzeltmenin buyuklugunu
  sinirlamak gibi bir onlem 3c'deki capraz dogrulamayla sinanmalidir (test yilina gore ayarlanmamalidir).
- 28 Temmuz'daki bulgu sicak gunlerin genelinde tekrarlanmiyorsa, o gune ozgu bir durumdur.""")

       gun model_a  gfs_ham_C  gfs_duzeltilmis_C  era5_C  a_mape_%  b_mape_%  plan_mape_%  a_sapma_%  b_sapma_%  plan_sapma_%
2026-07-22 secilen      29.10              29.14   28.65      0.74      0.38         1.38       0.60      -0.06         -0.24
2026-08-12 secilen      28.29              27.99   28.27      0.45      0.39         1.93       0.17       0.18         -0.60
2026-07-21 secilen      28.56              28.59   28.25      1.26      0.43         2.04       1.25       0.24          0.14
2026-08-06 secilen      27.86              27.60   28.08      0.40      0.34         1.50      -0.09      -0.19          0.44
2026-08-05 secilen      27.95              27.74   28.07      0.83      0.53         2.18       0.70       0.22          1.25
2026-07-20 secilen      28.13              28.18   28.06      0.63      0.31         1.89       0.61       0.09          1.12
2026-08-04 secilen      28.04              27.94   28.00      0.51      0.52         1.75      -0.01      -0.04       

### E3. Aynı örüntü çapraz doğrulama yıllarında da var mı?

E ve E2 bulguları test yılından geliyor. Bir önlem düşünmeden önce, aynı örüntünün modelin seçildiği **dört çapraz doğrulama yılında** (2021-22 → 2024-25) da görülüp görülmediği kontrol ediliyor. Kaynak `metrics/tr_cv_katman_tahminleri.csv`: her katmanda, o yıldan önceki veriyle eğitilmiş modelin tahminleri.

Günler, ağırlıklı ERA5 günlük ortalama sıcaklığına göre dört dilime ayrılıyor: <20°C, 20-24°C, 24-27°C, ≥27°C. Her dilimde modelin ve resmî planın ortalama sapması ve MAPE'si hesaplanıyor.

**Karar kuralı (önceden yazıldı):** ≥27°C diliminde modelin sapması, **dört katmanın en az üçünde** hem pozitif hem de planınkinden büyükse, örüntü doğrulanmış sayılır. Bu durumda:
- Agent'a bir "sıcak gün" uyarısı eklenmesi gerekçelendirilmiş olur.
- Olası bir düzeltme (plana yapılan düzeltmenin sıcakta sınırlanması) yalnızca bu katmanlar üzerinde denenir.

Aksi hâlde bulgu yalnızca test yılına özgü bir gözlem olarak raporlanır.

In [10]:
kt = pd.read_csv(os.path.join(KOK, "metrics", "tr_cv_katman_tahminleri.csv"))
kt["date"] = pd.to_datetime(kt["date"])
kt["gun_t"] = kt["date"].dt.tz_localize(None).dt.normalize()
kt["t_gun"] = kt["gun_t"].map(zt.df.groupby("gun_t")["om_era5"].mean())
SEC, PLAN = zt.meta_secilen["model"], "Resmi Yuk Tahmin Plani"
for ad, kol in [("model", SEC), ("plan", PLAN)]:
    kt[f"sap_{ad}"] = (kt[kol] - kt["gercek"]) / kt["gercek"] * 100
kt["dilim"] = pd.cut(kt["t_gun"], [-50, 20, 24, 27, 50], labels=["<20", "20-24", "24-27", ">=27"], right=False)
ozet_d = kt.groupby("dilim", observed=True).agg(saat=("gercek", "size"),
                                                model_sapma_pct=("sap_model", "mean"), plan_sapma_pct=("sap_plan", "mean"),
                                                model_mape_pct=("sap_model", lambda s: s.abs().mean()),
                                                plan_mape_pct=("sap_plan", lambda s: s.abs().mean()))
print("Dort katman birlikte, sicaklik dilimine gore (%):")
print(ozet_d.round(2).to_string())
kat = kt[kt["dilim"] == ">=27"].groupby("katman").agg(saat=("gercek", "size"), model_sapma=("sap_model", "mean"),
                                                      plan_sapma=("sap_plan", "mean"))
kat["model_yukari_ve_plandan_buyuk"] = (kat["model_sapma"] > 0) & (kat["model_sapma"] > kat["plan_sapma"])
print("\n>=27 C dilimi, katman bazinda (%):")
print(kat.round(2).to_string())
n_ok = int(kat["model_yukari_ve_plandan_buyuk"].sum())
dogrulandi = n_ok >= 3
print(f"\nKarar: {n_ok}/{len(kat)} katmanda model sicakta yukari sapiyor ve plandan fazla -> "
      + ("ORUNTU DOGRULANDI (capraz dogrulama yillarinda da var)." if dogrulandi else "oruntu capraz dogrulamada dogrulanmadi; test yilina ozgu gozlem."))
ozet_d.round(3).to_csv(os.path.join(KOK, "metrics", "tr_cv_sicaklik_sapmasi.csv"))

Dort katman birlikte, sicaklik dilimine gore (%):
        saat  model_sapma_pct  plan_sapma_pct  model_mape_pct  plan_mape_pct
dilim                                                                       
<20    22236            -0.63           -0.27            2.42           2.82
20-24   4224            -1.09           -0.69            2.48           2.89
24-27   5112            -0.34           -1.62            2.81           3.31
>=27    2928            -0.72           -3.68            2.26           4.11

>=27 C dilimi, katman bazinda (%):
        saat  model_sapma  plan_sapma  model_yukari_ve_plandan_buyuk
katman                                                              
1        216        -1.21       -2.18                          False
2        768        -1.02       -3.81                          False
3        864        -0.78       -4.13                          False
4       1080        -0.38       -3.54                          False

Karar: 0/4 katmanda model sicakta yuk